In [1]:
import pandas as pd
import psycopg2
import os
from dotenv import load_dotenv


In [2]:
# 1. Load environment variables from .env file
try:
    load_dotenv(dotenv_path='/home/kiet/gkinhere/jobs-pipeline-analysis/.env')
    print("Environment variables loaded successfully.")
except Exception as e:
    print(f"Error loading environment variables: {e}")

# 2. Connect to the PostgreSQL database using environment variables
db_name = os.getenv('DB_NAME')
db_user = os.getenv('DB_USER')
db_password = os.getenv('DB_PASSWORD')
db_host = 'localhost' # tuy là chạy docker nhưng ở đây ko nằm trong network container nên ko dùng tên container
                      # và đây là máy host nên vẫn dùng localhost được
db_port = 5433 # port mình đã map bằng docker compose là 5433 và mình là máy host nên có thể dùng luôn
print(f"Connecting to database {db_name} at {db_host}:{db_port} as user {db_user}")

# 3. Establish the database connection
conn = None # Initialize connection variable
try:
    print("Connecting to the database...")
    conn = psycopg2.connect(
        dbname=db_name,
        user=db_user,
        password=db_password,
        host=db_host,
        port=db_port
    )
    print("Connection established successfully.")
except Exception as e:
    print(f"Error connecting to the database: {e}")

Environment variables loaded successfully.
Connecting to database topcvdb at localhost:5433 as user postgres
Connecting to the database...
Connection established successfully.


In [3]:
# Cell 3: Đọc dữ liệu từ PostgreSQL vào Pandas DataFrame

# Kiểm tra xem kết nối 'conn' từ Cell 2 có thực sự tồn tại và hoạt động không
if conn:
    # 1. Định nghĩa câu lệnh SQL (Query) để lấy dữ liệu
    query = "SELECT * FROM topcv_jobs_detailed WHERE status = 'completed' "
    print("Sending query to the database...")
    print(f"Query: {query}")
    
    # 2. Sử dụng hàm pd.read_sql() của Pandas để thực thi query và tải dữ liệu
    # Hàm này là "cây cầu thần kỳ" nối SQL và Pandas
    # Nó nhận vào câu lệnh SQL và đối tượng kết nối 'conn'
    df = pd.read_sql(query, conn)
    print("\n Data loaded successfully into DataFrame")
    # len(df) trả về số lượng hàng trong DataFrame
    print(f"Number of rows in DataFrame: {len(df)}")
    
    # 3. Đóng kết nối sau khi đã lấy xong dữ liệu. Đây là một thói quen tốt.
    conn.close()
    print("Database connection closed.")
    
    # 4. Hiển thị một vài dòng đầu tiên của DataFrame để kiểm tra dữ liệu
    print("\nFirst few rows of the DataFrame:")
    print(df.head())
else:
    print("No connection to the database. Cannot load data into DataFrame.")


Sending query to the database...
Query: SELECT * FROM topcv_jobs_detailed WHERE status = 'completed' 

 Data loaded successfully into DataFrame
Number of rows in DataFrame: 926
Database connection closed.

First few rows of the DataFrame:
   job_id source_site                                          job_title  \
0     196       topcv  Nhà Thiết Kế & Phát Triển Mẫu (Chuyên Về Thời ...   
1       7       topcv  Nhân Viên Tư Vấn Sản Phầm/Sales/CSKH/Thu Nhập ...   
2      11       topcv  Kế Toán Nội Bộ (Mức Lương Từ 11 - 15 Triệu/Thá...   
3      15       topcv  Nhân Viên Tư Vấn Chăm Sóc Khách Hàng - Lương C...   
4      20       topcv  Idol Live TikTok (Fulltime/Parttime) | Thu Nhậ...   

                                             job_url scrape_date     status  \
0  https://www.topcv.vn/viec-lam/nha-thiet-ke-pha...  2025-07-02  completed   
1  https://www.topcv.vn/viec-lam/nhan-vien-tu-van...  2025-07-02  completed   
2  https://www.topcv.vn/viec-lam/ke-toan-noi-bo-m...  2025-07-02  c

/tmp/ipykernel_67705/3335806790.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
